# Intercept-Policy Experiment on GOLDEN (Skeleton-Driven)

Test whether the Stage 4 LLM thrashing observed in the 2026-04-16 trace can be eliminated by changing the *default* model-spec policy rather than by hard-sparsity drift, hard-coding a fix in the agent loop, or a structural reparameterisation.

## How elicitation is mimicked

1. Start from `derive_deterministic_spec(causal_spec)` — the structural skeleton, **identical** to what the Stage 4 agent sees on turn 1, before any prior elicitation.
2. Use the megaprompt's accepted likelihoods (distribution + link decisions) as the resolved choices for ambiguous indicators — these are deterministic data-driven and don't represent thrashing.
3. Author **simulated turn-1 priors** deterministically per parameter role + indicator empirical statistics. Wide where it's reasonable to be wide (intercepts, hyperparameters), informed by data scale where the parameter has a clear data-side anchor.
4. Run `validate_assembly` under different policy combinations holding the priors fixed.

Compared to the previous run (which lifted the LLM's converged tight priors), this asks the right question: **at turn 1, before any iteration, which policy combination makes prior-predictive validation pass?**

## Configurations

| Config | `observation_intercept_policy` | `equilibrium_forcing` | log-link manifest_mean σ | cint priors |
|---|---|---|---|---|
| **A — Wide turn-1** | `free` | `False` | 1.0 | n/a |
| **B — Deployed policy** | `fixed` | `False` | n/a | n/a |
| **C — Equilibrium forcing** | `fixed` | `True` | n/a | `Normal(0, 0.3)` per construct |
| **D — Tight manifest_means** | `free` | `False` | 0.3 | n/a |

All four use the same naive turn-1 priors for every other parameter (rho, beta, sigma, t0, etc.).

In [ ]:
from __future__ import annotations

import copy
import json
import math
import sys
from pathlib import Path

import polars as pl

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'apps').exists():
    for parent in REPO_ROOT.parents:
        if (parent / 'apps').exists():
            REPO_ROOT = parent
            break

sys.path.insert(0, str(REPO_ROOT / 'apps/data-pipeline/src'))
from causal_ssm_agent.flows.stages.stage4.agentic.stage4_skeleton import derive_deterministic_spec
from causal_ssm_agent.flows.stages.stage4.assembly import validate_assembly
from causal_ssm_agent.flows.stages.stage4.model_spec_decisions import validate_model_spec_decisions_dict

GOLDEN = REPO_ROOT / 'data/.private/GOLDEN/run'
megaprompt = json.loads((GOLDEN / 'stage-4-megaprompt.json').read_text())
causal_spec = json.loads((GOLDEN / 'stage-1b.json').read_text())['causal_spec']
indicator_audits = json.loads((GOLDEN / 'stage-3.json').read_text())['indicators']
data_for_model = pl.read_parquet(GOLDEN / 'stage2-model-data.parquet')

# Build the deterministic skeleton — what Stage 4 agent sees on turn 1.
skeleton = derive_deterministic_spec(causal_spec)

# Lift distribution-choice decisions from the megaprompt's accepted likelihoods.
# These are deterministic data-driven; the LLM doesn't iterate on them.
mp_likelihoods = {lh['variable']: lh for lh in megaprompt['accepted']['model_spec']['likelihoods']}
distribution_choices = []
for ambiguous in skeleton.ambiguous_indicators:
    name = ambiguous['variable']
    if name in mp_likelihoods:
        distribution_choices.append({
            'variable': name,
            'distribution': mp_likelihoods[name]['distribution'],
            'link': mp_likelihoods[name]['link'],
            'reasoning': 'lifted from megaprompt accepted likelihood',
        })

# Build empirical statistics per indicator from the audits.
def empirical_stats(name: str) -> dict[str, float]:
    audit = indicator_audits.get(name, {}) or {}
    profile = audit.get('profile') or {}
    out = {}
    for key in ('mean', 'std', 'data_mean'):
        v = audit.get(key) if isinstance(audit, dict) else None
        if v is None:
            v = profile.get(key)
        if v is not None:
            try:
                out[key] = float(v)
            except (TypeError, ValueError):
                continue
    out.setdefault('mean', 0.0)
    out.setdefault('std', 1.0)
    return out


print(f'Skeleton parameters: {len(skeleton.all_params)} '
      f'(includes {len(skeleton.loading_params)} loading params)')
print(f'Resolved likelihoods: {len(skeleton.resolved_likelihoods)}')
print(f'Ambiguous indicators (need distribution_choice): {len(skeleton.ambiguous_indicators)}')
print(f'Distribution choices lifted from megaprompt: {len(distribution_choices)}')

In [ ]:
# --- Helpers ---

def build_model_spec(*, initialization_policy: str, observation_intercept_policy: str,
                     equilibrium_forcing: bool) -> dict:
    """Build a model_spec from the skeleton + policy choices."""
    decisions = {
        'initialization_policy': initialization_policy,
        'observation_intercept_policy': observation_intercept_policy,
        'equilibrium_forcing': equilibrium_forcing,
        'distribution_choices': distribution_choices,
    }
    spec, errors = validate_model_spec_decisions_dict(
        decisions,
        resolved_likelihoods=skeleton.resolved_likelihoods,
        ambiguous_indicators=skeleton.ambiguous_indicators,
        parameters=skeleton.all_params,
    )
    if errors or spec is None:
        raise ValueError(errors)
    return spec.model_dump(mode='json')


def _prior(parameter: str, distribution: str, params: dict, reasoning: str) -> dict:
    return {
        'parameter': parameter,
        'distribution': distribution,
        'params': params,
        'sources': [],
        'reasoning': reasoning,
        'reference_interval_days': None,
        'density_points': None,
    }


def author_turn1_priors(spec: dict, *, manifest_mean_sigma: float = 1.0,
                        cint_sigma: float = 0.3) -> dict:
    """Author plausible turn-1 priors for every active parameter.

    All priors are wide where reasonable, data-anchored only for parameters
    with a natural data-side anchor (manifest means, initial-state means).
    """
    likelihoods = {lh['variable']: lh for lh in spec['likelihoods']}
    priors: dict[str, dict] = {}

    for p in spec['parameters']:
        name = p['name']
        role = p['role']
        if role == 'ar_coefficient':
            priors[name] = _prior(name, 'Beta', {'alpha': 2.0, 'beta': 2.0},
                                  'naive AR(1) persistence prior on (0,1)')
        elif role == 'fixed_effect':
            priors[name] = _prior(name, 'Normal', {'mu': 0.0, 'sigma': 0.5},
                                  'naive cross-effect, weakly informative around zero')
        elif role == 'residual_sd':
            priors[name] = _prior(name, 'Gamma', {'concentration': 2.0, 'rate': 1.0},
                                  'naive residual SD, unit-scale positive')
        elif role == 'state_intercept':
            priors[name] = _prior(name, 'Normal', {'mu': 0.0, 'sigma': cint_sigma},
                                  f'naive CINT prior, sigma={cint_sigma}')
        elif role == 'initial_state_mean':
            priors[name] = _prior(name, 'Normal', {'mu': 0.0, 'sigma': 1.0},
                                  'naive initial-state mean, weakly informative')
        elif role == 'initial_state_sd':
            priors[name] = _prior(name, 'Gamma', {'concentration': 2.0, 'rate': 1.0},
                                  'naive initial-state SD')
        elif role == 'static_state_sd':
            priors[name] = _prior(name, 'Gamma', {'concentration': 2.0, 'rate': 1.0},
                                  'naive static-factor SD')
        elif role == 'observation_intercept':
            indicator = name.removeprefix('manifest_mean_')
            link = likelihoods.get(indicator, {}).get('link', 'identity')
            stats = empirical_stats(indicator)
            if link == 'log':
                emp = stats.get('mean', 1.0) or 1e-6
                mu = math.log(emp) if emp > 0 else 0.0
                priors[name] = _prior(name, 'Normal',
                                      {'mu': mu, 'sigma': manifest_mean_sigma},
                                      f'log-link intercept, log(mean)={mu:.2f}, sigma={manifest_mean_sigma}')
            else:
                mu = stats.get('mean', 0.0)
                sigma = max(stats.get('std', 1.0), 1.0)
                priors[name] = _prior(name, 'Normal',
                                      {'mu': mu, 'sigma': sigma},
                                      f'identity-link intercept, mean={mu:.2f}, sigma={sigma:.2f}')
        elif role == 'measurement_error_sd':
            priors[name] = _prior(name, 'Gamma', {'concentration': 2.0, 'rate': 1.0},
                                  'naive measurement-error SD')
        elif role == 'observation_hyperparameter_positive':
            if name == 'obs_r' or name.startswith('obs_r'):
                priors[name] = _prior(name, 'Gamma', {'concentration': 2.0, 'rate': 0.5},
                                      'naive negbin dispersion')
            elif name == 'obs_shape' or name.startswith('obs_shape'):
                priors[name] = _prior(name, 'Gamma', {'concentration': 2.0, 'rate': 1.0},
                                      'naive gamma shape')
            else:
                priors[name] = _prior(name, 'Gamma', {'concentration': 2.0, 'rate': 1.0},
                                      'naive positive observation hyperparameter')
        elif role == 'correlation' or role == 'initial_state_correlation':
            priors[name] = _prior(name, 'Uniform', {'lower': -0.9, 'upper': 0.9},
                                  'naive correlation prior')
        elif role == 'loading':
            constraint = p.get('constraint', 'positive')
            mu = 1.0 if constraint != 'negative' else -1.0
            priors[name] = _prior(name, 'Normal', {'mu': mu, 'sigma': 0.3},
                                  'naive loading prior centered at 1')
        else:
            print(f'  WARNING: unhandled role {role!r} for {name}')
    return priors


def _sensitivity_fail_count(validation):
    payload = validation.sensitivity_payload or {}
    weak = payload.get('weak_directions') or []
    return sum(1 for d in weak if isinstance(d, dict) and d.get('status') == 'fail')


def summarise(label: str, validation):
    failing = [d.model_dump(mode='json') for d in validation.diagnostics if not d.is_valid]
    overflow_count = sum(
        1 for d in failing if 'overflow' in (d.get('issue') or '').lower()
    )
    return {
        'label': label,
        'is_valid': validation.is_valid,
        'compile_ok': validation.compile_ok,
        'compile_error': (validation.compile_error or '')[:300] if validation.compile_error else None,
        'pp_checked': validation.pp_checked,
        'pp_valid': validation.pp_valid,
        'sensitivity_valid': validation.sensitivity_valid,
        'n_pp_failures': len(failing),
        'n_overflow_failures': overflow_count,
        'n_sens_failing_dirs': _sensitivity_fail_count(validation),
        'failing_codes': sorted({d.get('code') for d in failing if d.get('code')}),
    }


def print_summary(s):
    print(f"  is_valid                : {s['is_valid']}")
    print(f"  compile_ok              : {s['compile_ok']}")
    if s['compile_error']:
        print(f"  compile_error           : {s['compile_error']}")
    print(f"  pp_checked / pp_valid   : {s['pp_checked']} / {s['pp_valid']}")
    print(f"  sensitivity_valid       : {s['sensitivity_valid']}")
    print(f"  n_pp_failures           : {s['n_pp_failures']}")
    print(f"  n_overflow_failures     : {s['n_overflow_failures']}")
    print(f"  n_sens_failing_dirs     : {s['n_sens_failing_dirs']}")
    print(f"  failing_codes           : {s['failing_codes']}")

## Config A — Thrashing reproduction (FREE intercepts, broad σ=1.0)

In [ ]:
spec_A = build_model_spec(
    initialization_policy='stationary',
    observation_intercept_policy='free',
    equilibrium_forcing=False,
)
priors_A = author_turn1_priors(spec_A, manifest_mean_sigma=1.0)
val_A = validate_assembly(spec_A, priors_A, data_for_model, indicator_audits, causal_spec)
summary_A = summarise('A — wide manifest_means', val_A)
print('Config A — Wide turn-1 (FREE, sigma=1.0)')
print_summary(summary_A)

## Config B — Deployed config (FIXED intercepts, no CINT)

In [ ]:
spec_B = build_model_spec(
    initialization_policy='stationary',
    observation_intercept_policy='fixed',
    equilibrium_forcing=False,
)
priors_B = author_turn1_priors(spec_B, manifest_mean_sigma=1.0)
val_B = validate_assembly(spec_B, priors_B, data_for_model, indicator_audits, causal_spec)
summary_B = summarise('B — deployed', val_B)
print('Config B — Deployed policy (FIXED manifest_means)')
print_summary(summary_B)

## Config C — Equilibrium forcing (CINT carries baseline, manifest_means=0)

In [ ]:
spec_C = build_model_spec(
    initialization_policy='stationary',
    observation_intercept_policy='fixed',
    equilibrium_forcing=True,
)
priors_C = author_turn1_priors(spec_C, manifest_mean_sigma=1.0, cint_sigma=0.3)
val_C = validate_assembly(spec_C, priors_C, data_for_model, indicator_audits, causal_spec)
summary_C = summarise('C — equilibrium_forcing', val_C)
print('Config C — Equilibrium forcing (FIXED + CINT free)')
print_summary(summary_C)

## Config D — Tight-prior shortcut (FREE intercepts, σ=0.3)

In [ ]:
spec_D = build_model_spec(
    initialization_policy='stationary',
    observation_intercept_policy='free',
    equilibrium_forcing=False,
)
priors_D = author_turn1_priors(spec_D, manifest_mean_sigma=0.3)
val_D = validate_assembly(spec_D, priors_D, data_for_model, indicator_audits, causal_spec)
summary_D = summarise('D — tight manifest_means', val_D)
print('Config D — Tight manifest_means (FREE, sigma=0.3)')
print_summary(summary_D)

## Comparison

In [ ]:
rows = [summary_A, summary_B, summary_C, summary_D]
header_keys = ['label', 'is_valid', 'compile_ok', 'pp_checked', 'pp_valid', 'sensitivity_valid', 'n_pp_failures', 'n_overflow_failures', 'n_sens_failing_dirs']
widths = {k: max(len(k), max(len(str(r.get(k))) for r in rows)) for k in header_keys}
header = '  '.join(k.ljust(widths[k]) for k in header_keys)
print(header)
print('  '.join('-' * widths[k] for k in header_keys))
for r in rows:
    print('  '.join(str(r.get(k)).ljust(widths[k]) for k in header_keys))
print()
for r in rows:
    if r['failing_codes']:
        print(f"{r['label']:30s} failing codes: {r['failing_codes']}")
    if r.get('compile_error'):
        print(f"{r['label']:30s} compile_error: {r['compile_error']}")

## Diagnostic inspection

What does the failing PP diagnostic actually say? Which channels overflow, which parameters are cited, which compile site?

In [ ]:
diag = next(d for d in val_A.diagnostics if not d.is_valid)
print(f'parameter           : {diag.parameter}')
print(f'code                : {diag.code}')
print(f'origin              : {diag.origin}')
print(f'severity            : {diag.severity}')
print(f'failure_stage       : {diag.failure_stage}')
print(f'related_parameters  : {diag.related_parameters}')
print(f'compiled_site_name  : {diag.compiled_site_name}')
print(f'pathology kind      : {diag.pathology_certificate.kind if diag.pathology_certificate else None}')
print()
print('issue:')
print('  ' + (diag.issue or '').replace('\n', '\n  '))
print()
print('suggested_adjustment:')
print('  ' + (diag.suggested_adjustment or '').replace('\n', '\n  '))

## Single-surface ablation

Hold Config A's spec (`FREE` manifest_means, no equilibrium forcing) fixed. Tighten one prior surface at a time on top of the same naive turn-1 priors. Whichever ablation flips `pp_valid` to `True` identifies the responsible surface.

Tightening levels are deliberately aggressive (much tighter than realistic LLM priors) to give a definitive yes/no per surface. The point is attribution, not realism.

In [ ]:
def _tightened_priors(predicate, *, normal_sigma=None, gamma_concentration=None, gamma_rate=None):
    out = copy.deepcopy(priors_A)
    for name, prior in out.items():
        if not predicate(name, prior):
            continue
        params = prior['params']
        if normal_sigma is not None and 'sigma' in params:
            params['sigma'] = normal_sigma
        if gamma_concentration is not None and 'concentration' in params:
            params['concentration'] = gamma_concentration
            params['rate'] = gamma_rate if gamma_rate is not None else gamma_concentration
    return out


def is_role(role_prefix):
    return lambda name, prior: name.startswith(role_prefix)


# Predicates for each surface
is_beta = is_role('beta_')
is_sigma_residual = is_role('sigma_')              # residual_sd
is_manifest_mean = is_role('manifest_mean_')
is_t0_sd = is_role('t0_sd_')
is_static_sd = is_role('tau_')                     # static_state_sd uses tau_ prefix
is_obs_sd = is_role('obs_sd_')                     # measurement_error_sd
is_any_positive = lambda name, prior: prior['distribution'] == 'Gamma'


# Note: sigma_* and obs_sd_* and t0_sd_* and tau_* are all Gamma in our turn-1 priors.
# Aggressive Gamma tightening: concentration=50, rate=100 -> mean 0.5, var 0.005 (tiny).
GAMMA_TIGHT = dict(gamma_concentration=50.0, gamma_rate=100.0)

ABLATIONS = [
    ('baseline (turn-1, all wide)', copy.deepcopy(priors_A)),
    ('beta_* tightened (sigma=0.05)', _tightened_priors(is_beta, normal_sigma=0.05)),
    ('sigma_* (residual_sd) tightened', _tightened_priors(is_sigma_residual, **GAMMA_TIGHT)),
    ('manifest_mean_* tightened (sigma=0.05)', _tightened_priors(is_manifest_mean, normal_sigma=0.05)),
    ('all positive priors tightened', _tightened_priors(is_any_positive, **GAMMA_TIGHT)),
    ('beta_* + sigma_* tightened', _tightened_priors(
        lambda n, p: is_beta(n, p) or is_sigma_residual(n, p),
        normal_sigma=0.05, **GAMMA_TIGHT,
    )),
    ('beta_* + sigma_* + manifest_mean_* tightened', _tightened_priors(
        lambda n, p: is_beta(n, p) or is_sigma_residual(n, p) or is_manifest_mean(n, p),
        normal_sigma=0.05, **GAMMA_TIGHT,
    )),
]

print(f"{'config':50s}  pp_valid  n_overflow  failing_codes")
print('-' * 95)
for label, priors in ABLATIONS:
    val = validate_assembly(spec_A, priors, data_for_model, indicator_audits, causal_spec)
    s = summarise(label, val)
    codes = ','.join(s['failing_codes']) or '-'
    print(f"{label:50s}  {str(s['pp_valid']):8s}  {s['n_overflow_failures']:^10d}  {codes}")

## Master sensitivity, PP gate bypassed

`validate_assembly` only runs the Jacobian sensitivity diagnostic when PP passes. Since master overflows on naive priors, sensitivity is never run — the `n_sens_failing_dirs: 0` we saw is a default, not a real evaluation.

This cell calls `compile_ssm_artifact` + `output_sensitivity_analysis` directly to get master's actual sensitivity spectrum on the same skeleton + priors as Config A on hard-sparsity. Apples-to-apples.

Some prior draws will produce nonfinite predictive moments (because of the same overflow); the sensitivity code skips them automatically. If too many are skipped, the analysis raises and we report that explicitly.

In [ ]:
from causal_ssm_agent.models.ssm_compiler import compile_ssm_artifact
from causal_ssm_agent.models.ssm_builder import prepare_model_runtime
from causal_ssm_agent.models.ssm.diagnostics import (
    OutputSensitivityUnsupportedError,
    get_stage4b_sweep_context,
    output_sensitivity_analysis,
)

compiled_A = compile_ssm_artifact(copy.deepcopy(spec_A), priors_A, causal_spec=causal_spec)
runtime_A = prepare_model_runtime(data_for_model=data_for_model, compiled_ssm=compiled_A)
try:
    sa_result = output_sensitivity_analysis(
        runtime_A.model,
        runtime_A.times,
        observations=runtime_A.observations,
        n_draws=8,
        seed=42,
        sweep_context=get_stage4b_sweep_context(runtime_A.model),
    )
except RuntimeError as exc:
    print(f'Sensitivity analysis failed: {exc}')
    sa_result = None
except OutputSensitivityUnsupportedError as exc:
    print(f'Sensitivity analysis unsupported: {exc}')
    sa_result = None

if sa_result is not None:
    norm_sv = sa_result.normalized_singular_values
    print(f'n_parameters         : {sa_result.n_parameters}')
    print(f'n_observations       : {sa_result.n_observations}')
    print(f'n_draws              : {sa_result.n_draws}')
    print(f'deficiency_count     : {sa_result.deficiency_count}')
    failing_count = sum(1 for d in sa_result.weak_directions if d.get('status') == 'fail')
    print(f'failing weak dirs    : {failing_count}')
    print()
    print('Normalized SV spectrum (sorted ascending, first 25):')
    for i, v in enumerate(sorted(norm_sv)[:25]):
        print(f'  [{i:3d}] {v:.6e}')
    print('...')
    print(f'max normalized SV    : {max(norm_sv):.4g}')
    print(f'min normalized SV    : {min(norm_sv):.6e}')
    print()
    print('Per-parameter top 8 weakest (by normalized_effective_sv):')
    per_param = sorted(sa_result.per_parameter, key=lambda p: p.get('normalized_effective_sv', 1e18))
    for p in per_param[:8]:
        print(f"  {p['parameter']:35s} norm_eff_sv={p['normalized_effective_sv']:.4g} status={p['normalized_sv_status']}")